# 01 · What an agent is: a model in a loop
Level L1 · Part 1 · core · ~20 min · builds on nothing (first lesson)

## Goal
By the end of this lesson you can explain, and show with a running toy, how an **agent**
differs from a single model call: it chooses a tool, sees the result, and decides when it is done.

## The idea
A language model only turns text into more text. Ask it "How many of my notes mention Mars?"
and it can only guess, because it has never seen your notes.
An **agent** wraps the model in a loop. Each turn the model either asks for a **tool**
(a normal Python function, like "search my notes") or gives a final answer.
Your code runs the tool, shows the model the result, and asks again.
The loop stops when the model says it is done.

Analogy: a smart colleague on the phone who cannot see your desk. They cannot count your notes,
but they can say "search for Mars and read me what you find", then answer.

## Picture
```mermaid
flowchart LR
    Q[Question] --> M[Model]
    M -->|asks for a tool| T[Run tool: search_notes]
    T -->|result added to history| M
    M -->|final answer| A[Answer]
```

## Already know this?
This is the first lesson of Part 1. If you can answer all of these, skip to lesson 06 (the Part's checkpoint).
1. What does an agent do that a single model call cannot?
2. What does "next-token prediction" mean, and how does a model produce a whole sentence from it?
3. Is one token the same as one word? Why does it matter for cost?
4. What does raising the temperature change?
5. What is the context window, and what happens when a conversation outgrows it?

> **Answers:** 1) it calls tools, reads their results and decides when to stop, in a loop.
> 2) the model scores every possible next token; it picks one, appends it, and repeats until a stop condition.
> 3) no, a word can be several tokens; you pay and are limited per token.
> 4) it flattens the probabilities, so less likely tokens are picked more often.
> 5) the maximum tokens (input + output) per call; older text must be dropped or summarised, and quality drops as it fills.

## Step by step
### 1. Scout's notes
Our running example is **Scout**, a research assistant over your local notes.
Here are six short notes. In later lessons they grow into a folder of files.

In [1]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
QUESTION = "How many notes mention Mars?"
print(len(NOTES), "notes loaded")
print("Question:", QUESTION)

6 notes loaded
Question: How many notes mention Mars?


### 2. A plain model call
We do not need a real model yet. A **fake model** is a small function that behaves like one:
text in, text out. This one does what any model must do without access to your notes: it guesses.
Real models do the same, just more fluently (lesson 07 makes the first real call).

In [2]:
def plain_model(prompt: str) -> str:
    """Text in, text out. It has no way to look at NOTES."""
    return "I can't see your notes, but probably a few of them mention Mars."

print(plain_model(QUESTION))

I can't see your notes, but probably a few of them mention Mars.


The answer is fluent but useless: no number. Nothing in a single call lets the model read `NOTES`.

### 3. A tool
A **tool** is an ordinary function your code is willing to run on the model's behalf.
Scout's first tool searches the notes for a word (case-insensitive) and returns the matches.

In [3]:
def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

hits = search_notes("mars")
for note in hits:
    print(note["id"], note["text"])

1 Mars has two small moons, Phobos and Deimos.
3 A day on Mars lasts about 24 hours and 39 minutes.
5 The rover Perseverance landed on Mars in February 2021.


We register it in a dictionary, so the loop can look up a tool by the name the model asks for.

In [4]:
TOOLS = {"search_notes": search_notes}
print("Tools Scout can use:", list(TOOLS))

Tools Scout can use: ['search_notes']


### 4. A model that can ask for a tool
Now the fake model reads the whole **history** (a list of messages) and returns one of two things:
- `{"type": "tool_call", "name": ..., "args": ...}`: "please run this tool for me", or
- `{"type": "answer", "text": ...}`: "I am done".

It is scripted, but it really *reads* the tool result to build its answer. Real models return
the same two shapes; lesson 22 shows the real format.

In [5]:
def scripted_model(history: list[dict]) -> dict:
    last = history[-1]
    if last["role"] == "user":                       # new question: look it up first
        topic = last["content"].rstrip("?").split()[-1]   # "...mention Mars?" -> "Mars"
        return {"type": "tool_call", "name": "search_notes", "args": {"query": topic}}
    if last["role"] == "tool":                       # saw a result: answer from it
        ids = [note["id"] for note in last["content"]]
        return {"type": "answer", "text": f"Notes that mention it: {len(ids)} (ids {ids})."}
    raise ValueError(f"unexpected role {last['role']!r}")

print(scripted_model([{"role": "user", "content": QUESTION}]))

{'type': 'tool_call', 'name': 'search_notes', 'args': {'query': 'Mars'}}


On a fresh question it asks for `search_notes` with `query="Mars"`. It does not run the tool itself:
a model can only *ask*. Running it is our job.

### 5. The agent loop
The **agent loop** is the few lines that connect the two: call the model; if it asks for a tool,
run the tool and add the result to the history; repeat until it answers.
`max_steps` stops a model that never finishes (lesson 30 covers stopping properly).

In [6]:
def run_agent(question: str, model, tools: dict, max_steps: int = 5) -> str:
    history = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = model(history)
        if reply["type"] == "answer":
            print(f"step {step}: model answers")
            return reply["text"]
        result = tools[reply["name"]](**reply["args"])          # run the requested tool
        print(f"step {step}: model calls {reply['name']}({reply['args']}) -> {len(result)} results")
        history.append({"role": "assistant", "content": reply})
        history.append({"role": "tool", "content": result})     # model sees it next turn
    raise RuntimeError(f"agent did not finish in {max_steps} steps")

print("Answer:", run_agent(QUESTION, scripted_model, TOOLS))

step 1: model calls search_notes({'query': 'Mars'}) -> 3 results
step 2: model answers
Answer: Notes that mention it: 3 (ids [1, 3, 5]).


Two steps: the model asked for a search, then answered from what it saw. The answer has a real
number, 3, which matches the three Mars notes printed in step 3.

### 6. Same loop, new questions
Nothing in the loop is about Mars. The model picks the tool arguments from the question,
and the loop just follows. Try two more questions, including one with no matches.

In [7]:
for q in ["How many notes mention Saturn?", "How many notes mention Pluto?"]:
    print(q)
    print("  Answer:", run_agent(q, scripted_model, TOOLS))

How many notes mention Saturn?
step 1: model calls search_notes({'query': 'Saturn'}) -> 1 results
step 2: model answers
  Answer: Notes that mention it: 1 (ids [6]).
How many notes mention Pluto?
step 1: model calls search_notes({'query': 'Pluto'}) -> 0 results
step 2: model answers
  Answer: Notes that mention it: 0 (ids []).


Pluto gives 0: a correct answer from evidence, not a guess. That is the difference between
a model call and an agent.

## At work
- **The loop is unit-tested with a scripted fake model.** Teams unit-test the loop (step limit,
  error recovery, parallel results) with a FakeModel like `scripted_model`, so tests need no key and
  never flake. → lesson 46
- **Loops that never stop cost real money.** Production agents cap steps, tokens and cost per run;
  a model that keeps asking for tools is a common failure. → lessons 30 and 54
- **Tool results are untrusted input.** A note or web page can contain "ignore your instructions";
  companies keep results inside tool-result messages and give tools least privilege. → lesson 51
- **Every step is logged:** model, tokens, stop reason, tool name and arguments, result size, so a bad
  answer can be traced to the step that caused it. → lesson 49

## What just happened
- A plain call (`plain_model`) could only guess, because the model cannot see `NOTES`.
- `search_notes` is a normal function; `TOOLS` maps the name the model asks for to that function.
- `scripted_model` returned a `tool_call` first, then an `answer` built from the tool result.
- `run_agent` ran the tool, appended the result to `history`, and called the model again: two steps, answer 3.
- The same loop answered Saturn (1) and Pluto (0) with no changes: the model chose, the loop obeyed.

## Common mistakes
**1. Running the tool but not adding the result to the history.** The model never sees the result,
so it asks for the same tool again and again until the step limit trips.

In [8]:
def run_agent_forgetful(question, model, tools, max_steps=5):
    history = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = model(history)
        if reply["type"] == "answer":
            return reply["text"]
        tools[reply["name"]](**reply["args"])   # result thrown away!
    raise RuntimeError(f"agent did not finish in {max_steps} steps")

Run it: the scripted model keeps asking for `search_notes`, because its last message is always the question.

In [9]:
run_agent_forgetful(QUESTION, scripted_model, TOOLS)

RuntimeError: agent did not finish in 5 steps

Fix: append the model's request and the tool result to `history`, as `run_agent` does. Without the
step limit this would have looped forever.

**2. The model asks for a tool you never registered.** Models sometimes invent tool names.

In [10]:
def confused_model(history):
    return {"type": "tool_call", "name": "web_search", "args": {"query": "Mars"}}

Our loop looks the name up in `TOOLS` blindly, so an invented name crashes it.

In [11]:
run_agent(QUESTION, confused_model, TOOLS)

KeyError: 'web_search'

Fix: only offer tools that exist, and check the name before calling. Lesson 28 turns this error into
a message the model can read and recover from, instead of a crash.

In [12]:
reply = confused_model([])
if reply["name"] not in TOOLS:
    print(f"Unknown tool {reply['name']!r}; available: {list(TOOLS)}")

Unknown tool 'web_search'; available: ['search_notes']


## Try it
1. Ask Scout "How many notes mention moons?" and print the answer. Why is it what it is?
2. Add a second tool `count_notes()` that returns all notes, and change a copy of `scripted_model` so
   that the question "How many notes are there?" calls `count_notes` instead of `search_notes`.
   Run it with `run_agent` without changing the loop.

In [13]:
# your code here

## Key terms
- **agent** — a model running in a loop that can call tools, read their results and decide when it is done.
- **tool** — an ordinary function your code runs when the model asks for it by name with arguments.
- **agent loop** — call the model; if it asks for a tool, run it and add the result to the history; repeat until it answers or a limit is hit.
- **fake model** — a scripted Python function that stands in for a real model, so the loop runs and tests without a key.